# Stage 4: ERAP2 Drug Design — DiffDock + RFdiffusion + Selectivity Screen

Three parts:
1. **DiffDock** — dock known inhibitors against ERAP2 to validate the structure
2. **RFdiffusion** — generate de novo protein binders (multiple lengths)
3. **Selectivity Screen** — counter-dock all candidates against ERAP1 to pick ERAP2-selective hits

**Target:** ERAP2 (Q6P179) active site: H370, E371, H374, E393
**Counter-target:** ERAP1 (Q9NZ08) — ~50% sequence identity, must NOT bind
**Structures:** AlphaFold DB (ERAP2 pLDDT 93.31, ERAP1 pLDDT 92.38)
**Experimental PDB:** ERAP2: 3SE6, 4E36, 5CU5 | ERAP1: 2YD0, 6RYB

**Goal:** Selective ERAP2 modulator — binds ERAP2 active site, does NOT bind ERAP1

In [ ]:
# Cell 1: Setup — download ERAP2 + ERAP1 + experimental structures
import os, requests, torch
os.chdir("/content")

structures = {
    "erap2.pdb": "https://alphafold.ebi.ac.uk/files/AF-Q6P179-F1-model_v6.pdb",
    "erap1.pdb": "https://alphafold.ebi.ac.uk/files/AF-Q9NZ08-F1-model_v6.pdb",
    "erap2_3SE6.pdb": "https://files.rcsb.org/download/3SE6.pdb",
    "erap1_2YD0.pdb": "https://files.rcsb.org/download/2YD0.pdb",
}

for filename, url in structures.items():
    if not os.path.exists(filename):
        resp = requests.get(url)
        if resp.ok:
            with open(filename, "w") as f:
                f.write(resp.text)
            print(f"Downloaded {filename}")
        else:
            print(f"Failed to download {filename}: {resp.status_code}")
    else:
        print(f"Already have {filename}")

print(f"\nGPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" if torch.cuda.is_available() else "")
print("\nERAP2 (target) + ERAP1 (counter-screen) structures ready.")

---
## Part 1: DiffDock — Dock Known Inhibitors Against ERAP2

Validates that our ERAP2 structure produces realistic binding poses.
If known inhibitors dock correctly, the structure is good for novel design.

In [ ]:
# Cell 2: Install DiffDock
if not os.path.exists("DiffDock"):
    !git clone https://github.com/gcorso/DiffDock.git
    %cd DiffDock
    !pip install -q -e . 2>&1 | tail -5
    %cd /content
else:
    print("DiffDock already cloned")

# Check install
try:
    import diffdock
    print("DiffDock imported")
except ImportError:
    print("DiffDock installed (may use CLI instead of import)")

In [ ]:
# Cell 3: Define ligands to dock
# Known aminopeptidase inhibitors from DrugBank + literature
ligands = {
    "Bestatin": "CC(O)C(=O)NC(CC1=CC=CC=C1)C(O)=O",
    "Tosedostat": "CC(C)CC(NC(=O)C(CC1=CC=CC=C1)OC(=O)C(C)(C)C)B(O)O",
    "Leucinethiol": "CC(CC)CS",
    "Phosphoramidon": "CC(CC1=CC=CC=C1)C(=O)NC(CC(=O)O)C(=O)NC(CC2=CC=CC=C2)C(=O)O",
}

# Save ligands as SDF for DiffDock
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors

os.makedirs("ligands", exist_ok=True)
valid_ligands = {}

for name, smiles in ligands.items():
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        print(f"  {name}: invalid SMILES, skipping")
        continue
    
    # Add hydrogens and generate 3D
    mol = Chem.AddHs(mol)
    AllChem.EmbedMolecule(mol, randomSeed=42)
    AllChem.MMFFOptimizeMolecule(mol)
    
    # Save as SDF
    sdf_path = f"ligands/{name}.sdf"
    writer = Chem.SDWriter(sdf_path)
    writer.write(mol)
    writer.close()
    
    mw = Descriptors.MolWt(Chem.MolFromSmiles(smiles))
    print(f"  {name}: MW={mw:.1f}, saved to {sdf_path}")
    valid_ligands[name] = sdf_path

# Also save SMILES as CSV for DiffDock's batch mode
with open("ligands/ligands.csv", "w") as f:
    f.write("complex_name,protein_path,ligand_description,protein_sequence\n")
    for name, smiles in ligands.items():
        if Chem.MolFromSmiles(smiles):
            f.write(f"{name},erap2.pdb,{smiles},\n")

print(f"\n{len(valid_ligands)} ligands ready for docking")

In [ ]:
# Cell 4: Run DiffDock
os.chdir("/content")

# DiffDock inference using CSV batch mode
!cd DiffDock && python -m inference \
    --protein_ligand_csv /content/ligands/ligands.csv \
    --out_dir /content/diffdock_results \
    --inference_steps 20 \
    --samples_per_complex 10 \
    --batch_size 4 \
    --no_final_step_noise \
    2>&1 | tail -30

# If the above fails, try the newer API:
# !cd DiffDock && python -m diffdock.inference \
#     --protein_ligand_csv /content/ligands/ligands.csv \
#     --out_dir /content/diffdock_results

In [ ]:
# Cell 5: Analyze DiffDock results
import glob

result_dirs = glob.glob("diffdock_results/*/")
print(f"DiffDock produced results for {len(result_dirs)} complexes")

for rdir in sorted(result_dirs):
    name = os.path.basename(rdir.rstrip("/"))
    sdfs = glob.glob(f"{rdir}/*.sdf")
    confs = glob.glob(f"{rdir}/*confidence*")
    
    print(f"\n{name}:")
    print(f"  Poses: {len(sdfs)}")
    
    # Read confidence scores if available
    if confs:
        with open(confs[0]) as f:
            scores = f.read().strip().split("\n")
        print(f"  Confidence scores: {scores[:5]}")
    
    # Read top pose
    if sdfs:
        top_sdf = sorted(sdfs)[0]
        mol = Chem.SDMolSupplier(top_sdf)[0]
        if mol:
            pos = mol.GetConformer().GetPositions()
            center = pos.mean(axis=0)
            print(f"  Top pose center: ({center[0]:.1f}, {center[1]:.1f}, {center[2]:.1f})")

if not result_dirs:
    print("No results yet. Check cell 4 for errors.")
    !ls diffdock_results/ 2>/dev/null || echo "No results directory"

# Cell 7: Run RFdiffusion binder design against ERAP2
# Multiple lengths — let binding affinity decide, not cost
os.chdir("/content")
os.makedirs("rfdiffusion_results", exist_ok=True)

# Design binders at 4 different lengths (10 each = 40 total)
lengths = [
    ("short", "30-40"),    # Cheap peptide synthesis ~$500
    ("medium", "50-60"),   # Moderate cost ~$800
    ("long", "70-90"),     # Recombinant expression ~$1000
    ("large", "90-110"),   # Maximum binding surface ~$1500
]

for label, length_range in lengths:
    print(f"\n{'='*60}")
    print(f"Designing {label} binders ({length_range} residues)...")
    print(f"{'='*60}")
    
    !cd RFdiffusion && python scripts/run_inference.py \
        inference.output_prefix=/content/rfdiffusion_results/erap2_{label} \
        inference.input_pdb=/content/erap2.pdb \
        'contigmap.contigs=[A370-393/0 {length_range}]' \
        'ppi.hotspot_res=[A370,A371,A374,A392,A393]' \
        inference.num_designs=10 \
        inference.ckpt_override_path=models/Complex_beta_ckpt.pt \
        2>&1 | tail -5

import glob
total = len(glob.glob("rfdiffusion_results/*.pdb"))
print(f"\nTotal binder designs generated: {total}")

In [ ]:
# Cell 6: Install RFdiffusion
os.chdir("/content")

if not os.path.exists("RFdiffusion"):
    !git clone https://github.com/RosettaCommons/RFdiffusion.git
    %cd RFdiffusion
    
    # Install SE3-Transformers dependency
    !pip install -q e3nn
    !pip install -q --no-cache-dir dgl -f https://data.dgl.ai/wheels/torch-2.5/cu121/repo.html 2>&1 | tail -3
    !pip install -q -e . 2>&1 | tail -5
    
    # Download model weights
    !mkdir -p models
    if not os.path.exists("models/Complex_beta_ckpt.pt"):
        print("Downloading RFdiffusion model weights...")
        !wget -q --show-progress -O models/Complex_beta_ckpt.pt \
            http://files.ipd.uw.edu/pub/RFdiffusion/Complex_beta_ckpt.pt
        !wget -q --show-progress -O models/Base_ckpt.pt \
            http://files.ipd.uw.edu/pub/RFdiffusion/Base_ckpt.pt
    
    %cd /content
else:
    print("RFdiffusion already installed")

!ls -lh RFdiffusion/models/*.pt 2>/dev/null || echo "No model weights found"

In [ ]:
# Cell 7: Run RFdiffusion binder design against ERAP2
os.chdir("/content")
os.makedirs("rfdiffusion_results", exist_ok=True)

# Design 10 binders targeting ERAP2 active site
# Hotspot residues: H370, E371, H374, K392 (variant), E393
# contigmap: target chain A, binder length 70-100 residues
!cd RFdiffusion && python scripts/run_inference.py \
    inference.output_prefix=/content/rfdiffusion_results/erap2_binder \
    inference.input_pdb=/content/erap2.pdb \
    'contigmap.contigs=[A370-393/0 70-100]' \
    'ppi.hotspot_res=[A370,A371,A374,A392,A393]' \
    inference.num_designs=10 \
    inference.ckpt_override_path=models/Complex_beta_ckpt.pt \
    2>&1 | tail -20

In [ ]:
# Cell 8: Analyze RFdiffusion results
import glob

binder_pdbs = sorted(glob.glob("rfdiffusion_results/*.pdb"))
print(f"Generated {len(binder_pdbs)} binder designs")

for pdb_path in binder_pdbs:
    name = os.path.basename(pdb_path)
    with open(pdb_path) as f:
        lines = f.readlines()
    atom_lines = [l for l in lines if l.startswith("ATOM")]
    chains = set(l[21] for l in atom_lines)
    residues_per_chain = {}
    for l in atom_lines:
        ch = l[21]
        resnum = int(l[22:26].strip())
        if ch not in residues_per_chain:
            residues_per_chain[ch] = set()
        residues_per_chain[ch].add(resnum)
    
    chain_info = ", ".join(f"{ch}:{len(res)} res" for ch, res in sorted(residues_per_chain.items()))
    print(f"  {name}: {len(atom_lines)} atoms, chains: {chain_info}")

if not binder_pdbs:
    print("No binder PDBs found. Check cell 7 for errors.")
    !ls rfdiffusion_results/ 2>/dev/null

In [ ]:
# Cell 9: Visualize best binder
!pip install -q py3Dmol
import py3Dmol

if binder_pdbs:
    best = binder_pdbs[0]  # First design
    with open(best) as f:
        pdb_data = f.read()
    
    view = py3Dmol.view(width=800, height=600)
    view.addModel(pdb_data, "pdb")
    
    # Target (chain A) in gray
    view.setStyle({"chain": "A"}, {"cartoon": {"color": "gray", "opacity": 0.7}})
    
    # Binder (chain B) in green
    view.setStyle({"chain": "B"}, {"cartoon": {"color": "green"}})
    
    # Hotspot residues in red
    for r in [370, 371, 374, 392, 393]:
        view.addStyle({"chain": "A", "resi": r}, {"stick": {"color": "red"}})
    
    view.zoomTo()
    print(f"Showing: {os.path.basename(best)}")
    print("Gray = ERAP2 target, Green = designed binder, Red sticks = active site")
    view.show()
else:
    print("No binder structures to visualize yet.")

---
## Part 3: Selectivity Screen — ERAP2 vs ERAP1

The key differentiator. ERAP1 has an inhibitor in Phase I/II already.
ERAP2-selective modulators are the unoccupied space.

We dock every RFdiffusion candidate against BOTH ERAP2 and ERAP1.
Keep only candidates that bind ERAP2 strongly but ERAP1 weakly or not at all.

In [ ]:
# Cell 10: Dock all RFdiffusion binders against ERAP1 (counter-screen)
import glob, os
os.chdir("/content")

binder_pdbs = sorted(glob.glob("rfdiffusion_results/*.pdb"))
print(f"Counter-screening {len(binder_pdbs)} binders against ERAP1...")

if binder_pdbs:
    # Create CSV for DiffDock batch docking against ERAP1
    # Extract binder chains from RFdiffusion output and dock against ERAP1
    os.makedirs("selectivity_results", exist_ok=True)
    
    # For protein-protein docking, we use the binder coordinates
    # and check interface energy against ERAP1 vs ERAP2
    
    # Simple approach: use BioPython to calculate interface contacts
    from Bio.PDB import PDBParser, NeighborSearch
    import numpy as np
    
    parser = PDBParser(QUIET=True)
    
    # Load ERAP1 structure
    erap1_struct = parser.get_structure("erap1", "erap1.pdb")
    erap1_atoms = [a for a in erap1_struct.get_atoms()]
    erap1_ns = NeighborSearch(erap1_atoms)
    
    # ERAP1 active site residues (equivalent to ERAP2)
    # ERAP1: H353, H357, E376 (zinc), E354 (catalytic)
    # These are the ERAP1 equivalents of ERAP2's H370, H374, E393, E371
    
    erap2_struct = parser.get_structure("erap2", "erap2.pdb")
    erap2_atoms = [a for a in erap2_struct.get_atoms()]
    erap2_ns = NeighborSearch(erap2_atoms)
    
    results = []
    
    for pdb_path in binder_pdbs:
        name = os.path.basename(pdb_path)
        struct = parser.get_structure(name, pdb_path)
        
        # Get binder chain atoms (chain B)
        binder_atoms = []
        target_atoms = []
        for chain in struct.get_chains():
            if chain.id == 'B':
                binder_atoms = list(chain.get_atoms())
            elif chain.id == 'A':
                target_atoms = list(chain.get_atoms())
        
        if not binder_atoms:
            continue
        
        # Count contacts with ERAP2 active site (within 5 Angstroms)
        erap2_contacts = 0
        for atom in binder_atoms:
            nearby = erap2_ns.search(atom.get_vector().get_array(), 5.0)
            for n in nearby:
                res = n.get_parent()
                if res.get_id()[1] in [370, 371, 374, 392, 393]:
                    erap2_contacts += 1
        
        # Count hypothetical contacts with ERAP1 active site
        # (approximate — checks if binder geometry would also fit ERAP1)
        erap1_contacts = 0
        for atom in binder_atoms:
            nearby = erap1_ns.search(atom.get_vector().get_array(), 5.0)
            for n in nearby:
                res = n.get_parent()
                if res.get_id()[1] in [353, 354, 357, 376]:
                    erap1_contacts += 1
        
        binder_length = len(set(a.get_parent().get_id()[1] for a in binder_atoms))
        selectivity = erap2_contacts / max(erap1_contacts, 1)
        
        results.append({
            "design": name,
            "binder_residues": binder_length,
            "erap2_active_site_contacts": erap2_contacts,
            "erap1_active_site_contacts": erap1_contacts,
            "selectivity_ratio": round(selectivity, 2),
        })
        
        status = "SELECTIVE" if selectivity > 3 else ("MODERATE" if selectivity > 1.5 else "NON-SELECTIVE")
        print(f"  {name}: ERAP2={erap2_contacts} ERAP1={erap1_contacts} ratio={selectivity:.1f} → {status}")
    
    import pandas as pd
    df = pd.DataFrame(results)
    df = df.sort_values("selectivity_ratio", ascending=False)
    df.to_csv("selectivity_results/selectivity_screen.csv", index=False)
    print(f"\nSaved selectivity results to selectivity_results/selectivity_screen.csv")
else:
    print("No binder PDBs found. Run RFdiffusion first (Part 2).")

In [ ]:
# Cell 11: Final ranking — top candidates for synthesis
import pandas as pd

if os.path.exists("selectivity_results/selectivity_screen.csv"):
    df = pd.read_csv("selectivity_results/selectivity_screen.csv")
    
    # Filter for selective binders
    selective = df[df["selectivity_ratio"] > 2.0].copy()
    selective = selective.sort_values("erap2_active_site_contacts", ascending=False)
    
    print("=" * 70)
    print("TOP ERAP2-SELECTIVE BINDER CANDIDATES")
    print("=" * 70)
    print(f"\nTotal designs: {len(df)}")
    print(f"ERAP2-selective (ratio > 2): {len(selective)}")
    
    if not selective.empty:
        print(f"\nTop candidates for synthesis:")
        print(selective.head(10).to_string(index=False))
        
        best = selective.iloc[0]
        print(f"\n*** LEAD CANDIDATE: {best['design']} ***")
        print(f"    Binder size: {best['binder_residues']} residues")
        print(f"    ERAP2 contacts: {best['erap2_active_site_contacts']}")
        print(f"    ERAP1 contacts: {best['erap1_active_site_contacts']}")
        print(f"    Selectivity: {best['selectivity_ratio']}x")
        
        # Estimate synthesis cost
        size = best['binder_residues']
        if size <= 40:
            cost = "$400-800 (peptide synthesis)"
        elif size <= 60:
            cost = "$800-1,500 (peptide synthesis)"
        else:
            cost = "$500-1,500 (recombinant expression)"
        print(f"    Est. synthesis cost: {cost}")
        
        print(f"\n    Next steps:")
        print(f"    1. File provisional patent ($75)")
        print(f"    2. Synthesize peptide")
        print(f"    3. SPR binding assay (ERAP2 + ERAP1 counter-screen)")
        print(f"    4. Enzyme inhibition assay")
    else:
        print("\nNo selective candidates found. Consider:")
        print("  - Redesigning with different hotspot residues")
        print("  - Targeting allosteric site instead of active site")
        print("  - Using Proteina-Complexa (higher hit rate) on Prime Intellect")
else:
    print("Run the selectivity screen (Cell 10) first.")